# 04 — Baseline ML Pipeline with Degree Features

## Movie Collaboration Network — Blockbuster Prediction

In [ ]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

In [ ]:
NEO4J_URI  = 'neo4j://127.0.0.1:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASS = 'Manuvamshi@12'
CLEAN_CSV  = '../data/movies_cleaned.csv'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print('Connected to Neo4j')

## Step 1 — Extract Degree Features from Neo4j

In [ ]:
DEGREE_QUERY = """
MATCH (m:Movie)
OPTIONAL MATCH (m)-[r]-()
WITH m, COUNT(r) AS movie_degree
OPTIONAL MATCH (a:Actor)-[:ACTED_IN]->(m)
WITH m, movie_degree, COUNT(DISTINCT a) AS actor_degree
OPTIONAL MATCH (m)-[:BELONGS_TO]->(g:Genre)
WITH m, movie_degree, actor_degree, COUNT(DISTINCT g) AS genre_degree
OPTIONAL MATCH (d:Director)-[:DIRECTED]->(m)
OPTIONAL MATCH (d)-[:DIRECTED]->(other_m:Movie)
WITH m, movie_degree, actor_degree, genre_degree,
     COUNT(DISTINCT other_m) AS director_total_movies, d
OPTIONAL MATCH (lead:Actor)-[:ACTED_IN]->(m)
OPTIONAL MATCH (lead)-[:ACTED_IN]->(other_m2:Movie)
WITH m, movie_degree, actor_degree, genre_degree, director_total_movies, d,
     COUNT(DISTINCT other_m2) AS actor_total_movies
OPTIONAL MATCH (d)-[:COLLABORATED_WITH]->(collab_a:Actor)
RETURN m.title                       AS node_id,
       movie_degree,
       actor_degree,
       genre_degree,
       director_total_movies,
       actor_total_movies,
       COUNT(DISTINCT collab_a)      AS director_collab_count
"""

with driver.session() as session:
    result = session.run(DEGREE_QUERY)
    degree_df = pd.DataFrame([dict(record) for record in result])

print(f'Extracted degree features for {len(degree_df):,} movie nodes')
print(f'Columns: {list(degree_df.columns)}')
degree_df.head()

In [ ]:
degree_df.describe()

## Step 2 — Build the Feature Matrix

In [ ]:
movies = pd.read_csv(CLEAN_CSV)
print(f'Loaded {len(movies):,} movies from {CLEAN_CSV}')
print(f'Columns: {list(movies.columns)}')
movies.head(3)

In [ ]:
BLOCKBUSTER_THRESHOLD = 100_000_000
movies['is_blockbuster'] = (movies['revenue'] > BLOCKBUSTER_THRESHOLD).astype(int)

print(f'Target distribution:')
print(movies['is_blockbuster'].value_counts())
print(f"\nClass balance: {movies['is_blockbuster'].mean():.1%} blockbusters")

In [ ]:
movies_for_merge = movies.rename(columns={'title': 'node_id'})
matrix = movies_for_merge.merge(degree_df, on='node_id', how='inner')

print(f'Feature matrix shape: {matrix.shape}')
print(f'Movies in CSV but missing from graph: {len(movies_for_merge) - len(matrix):,}')
matrix.head(3)

In [ ]:
numeric_cols = [
    'budget',
    'runtime',
    'release_year',
    'movie_degree',
    'actor_degree',
    'genre_degree',
    'director_total_movies',
    'actor_total_movies',
    'director_collab_count',
]

categorical_cols = [
    'original_language',
]

feature_cols = numeric_cols + categorical_cols

X = matrix[feature_cols].copy()
y = matrix['is_blockbuster'].copy()
node_ids = matrix['node_id'].copy()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
print('Missing values per column:')
print(X.isna().sum())
print(f'\nTotal rows with any missing: {X.isna().any(axis=1).sum()}')

mask = ~X.isna().any(axis=1)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)
node_ids = node_ids[mask].reset_index(drop=True)

print(f'\nAfter dropping NaN rows: X.shape={X.shape}, y.shape={y.shape}')

## Step 3 — Split FIRST, then build the Pipeline

In [ ]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, node_ids,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(f'Train: {X_train.shape},  positive rate = {y_train.mean():.3f}')
print(f'Test : {X_test.shape},  positive rate = {y_test.mean():.3f}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),                              numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'),        categorical_cols),
    ]
)

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
    )),
])

print('Pipeline built. Steps:')
for name, step in pipe.named_steps.items():
    print(f'  {name}: {type(step).__name__}')

## Fit and Evaluate

In [ ]:
pipe.fit(X_train, y_train)
print('Pipeline fitted on training data.')

In [ ]:
y_pred = pipe.predict(X_test)

print('=' * 60)
print('BASELINE METRICS')
print('=' * 60)
print(classification_report(y_test, y_pred,
                            target_names=['Not Blockbuster', 'Blockbuster']))

print('Confusion matrix:')
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['actual: Not BB', 'actual: BB'],
    columns=['pred: Not BB', 'pred: BB'],
)
print(cm_df)

In [ ]:
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='f1_macro', n_jobs=-1)
print(f'5-fold CV macro-F1: mean = {cv_scores.mean():.4f}  std = {cv_scores.std():.4f}')
print(f'Per-fold scores: {np.round(cv_scores, 4)}')

## Feature Importances

In [ ]:
ohe_cols = pipe.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = list(numeric_cols) + list(ohe_cols)

importances = pipe.named_steps['classifier'].feature_importances_

fi_df = pd.DataFrame({
    'feature':    all_feature_names,
    'importance': importances,
}).sort_values('importance', ascending=False).reset_index(drop=True)

GRAPH_FEATS = {'movie_degree', 'actor_degree', 'genre_degree',
               'director_total_movies', 'actor_total_movies', 'director_collab_count'}
TAB_NUM     = {'budget', 'runtime', 'release_year'}

def tag_family(name):
    if name in GRAPH_FEATS:
        return 'graph (degree)'
    if name in TAB_NUM:
        return 'tabular numeric'
    return 'tabular categorical'

fi_df['family'] = fi_df['feature'].apply(tag_family)
fi_df.head(15)

In [ ]:
summary = fi_df.groupby('family')['importance'].agg(['sum', 'mean', 'count'])
summary['sum_pct'] = (summary['sum'] / summary['sum'].sum() * 100).round(2)
print('Feature importance by family:')
print(summary)
print(f"\nGraph-derived features share: {summary.loc['graph (degree)', 'sum_pct']}%")

## Save the Baseline Matrix for S6

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

baseline_matrix = pd.concat([
    node_ids.rename('node_id'),
    X,
    y.rename('is_blockbuster'),
], axis=1)

out_path = '../data/baseline_feature_matrix.parquet'
baseline_matrix.to_parquet(out_path, index=False)
print(f'Saved baseline matrix to {out_path}')
print(f'Shape: {baseline_matrix.shape}')
print(f'Columns: {list(baseline_matrix.columns)}')

baseline_f1_macro = f1_score(y_test, y_pred, average='macro')
baseline_f1_pos   = f1_score(y_test, y_pred, pos_label=1)

with open('../data/baseline_metrics.txt', 'w') as f:
    f.write(f'baseline_f1_macro={baseline_f1_macro:.4f}\n')
    f.write(f'baseline_f1_blockbuster={baseline_f1_pos:.4f}\n')
    f.write(f'cv_f1_macro_mean={cv_scores.mean():.4f}\n')
    f.write(f'cv_f1_macro_std={cv_scores.std():.4f}\n')

print('\nBaseline metric persisted to ../data/baseline_metrics.txt')
print(f'  test macro-F1     = {baseline_f1_macro:.4f}')
print(f'  test blockbuster F1 = {baseline_f1_pos:.4f}')

In [ ]:
driver.close()
print('Neo4j driver closed.')